<a href="https://colab.research.google.com/github/mohdhi5253/SpendDNA-Financial-Transaction-Analysis/blob/main/SpendDNA_MohammadDhilawala.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SpendDNA - Financial Transaction Analysis

### Name : Mohammad Dhilawala
### Batch : June Batch

## Objective
To analyze six months of financial transaction data using Python, NumPy and Pandas by cleaning messy banking data, extracting vendors, categorizing expenses, detecting anomalies and identifying spending personality.

# Feature 1: Transaction Parser

In [76]:
# Import Libraries

import pandas as pd
import numpy as np

In [77]:
df = pd.read_csv("Data set for DADS June.csv")

print("Original Shape :", df.shape)
df.head()

Original Shape : (1328, 8)


,Date,Time,Description,Type,Amount,Balance,Mode,Ref
0,2024-01-01,03:11,AMAZON SELLER SVCS,Debit,₹2462,678275.0,UPI,TXN190872
1,01-Jan-24,05:44,BHIM-BMTC,DR,50.00,681007.0,UPI,TXN143064
2,01-Jan-24,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,CR,₹84728,484728.0,NEFT,TXN246316
3,2024-01-01,14:07,UPI-AMAN-8934@OKAXIS,Debit,₹1828,-748745.0,UPI,TXN569226
4,01 Jan 2024,14:23,BHIM-BLINKIT,Debit,270.00,680737.0,UPI,TXN968962


In [78]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1328 entries, 0 to 1327
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Date         1328 non-null   object 
 1   Time         1328 non-null   object 
 2   Description  1328 non-null   object 
 3   Type         1328 non-null   object 
 4   Amount       1328 non-null   object 
 5   Balance      1328 non-null   float64
 6   Mode         1328 non-null   object 
 7   Ref          1328 non-null   object 
dtypes: float64(1), object(7)
memory usage: 83.1+ KB


In [79]:
df.isnull().sum()

,0
Date,0
Time,0
Description,0
Type,0
Amount,0
Balance,0
Mode,0
Ref,0


In [80]:
df.describe(include="all")

,Date,Time,Description,Type,Amount,Balance,Mode,Ref
count,1328,1328,1328,1328,1328,1328.000000,1328,1328
unique,617,832,283,3,1141,NaN,4,1307
top,2024-02-07,16:19,POS SWIGGY BANGALORE,DR,"Rs. 15,000",NaN,UPI,TXN427270
freq,6,6,30,670,5,NaN,1299,2
mean,NaN,NaN,NaN,NaN,NaN,147805.330572,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,341059.809969,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,-769127.000000,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,-104585.750000,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,151650.000000,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,448324.250000,NaN,NaN


In [81]:
#Check Duplicate Rows
print("Duplicate Rows :", df.duplicated().sum())

Duplicate Rows : 18


In [82]:
#Remove Duplicates
df = df.drop_duplicates()

print("Shape After Removing Duplicates :", df.shape)

Shape After Removing Duplicates : (1310, 8)


In [83]:
#Convert Date Column
df["Date"] = pd.to_datetime(
    df["Date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)
print("Invalid Dates :", df["Date"].isna().sum())

Invalid Dates : 0


In [84]:
#Clean Amount Column
df["Amount"] = (
    df["Amount"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

In [85]:
df["Amount"] = pd.to_numeric(
    df["Amount"],
    errors="coerce"
)
print("Invalid Amounts :", df["Amount"].isna().sum())

Invalid Amounts : 0


In [86]:
#Standardize Transaction Type
df["Type"] = df["Type"].replace({
    "DR": "debit",
    "Debit": "debit",
    "CR": "credit",
    "Credit": "credit"
})

df["Type"] = df["Type"].str.lower()

df["Type"].value_counts()

,count
Type,
debit,1304
credit,6


In [87]:
#Handle Missing Mode Values
df["Mode"] = df["Mode"].replace("", np.nan)


In [88]:
#Remove Invalid Rows
df = df.dropna(subset=["Date", "Amount"])

print("Final Shape :", df.shape)

Final Shape : (1310, 8)


In [89]:
print("=" * 60)
print("TRANSACTION PARSER COMPLETED")
print("=" * 60)

print("Total Transactions :", len(df))
print("Duplicate Removed  :", 18)
print("Invalid Dates      :", df["Date"].isna().sum())
print("Invalid Amounts    :", df["Amount"].isna().sum())

TRANSACTION PARSER COMPLETED
Total Transactions : 1310
Duplicate Removed  : 18
Invalid Dates      : 0
Invalid Amounts    : 0


In [90]:
df_clean = df.copy()

# Feature 2 – Vendor Extractor

In [91]:
#Check Unique Descriptions
print("Total Unique Descriptions :", df['Description'].nunique())

df['Description'].unique()

Total Unique Descriptions : 283


array(['AMAZON SELLER SVCS', 'BHIM-BMTC',
       'NEFT-TECHCRUSH LABS-SALARY MAY24', 'UPI-AMAN-8934@OKAXIS',
       'BHIM-BLINKIT', 'BHIM ZEPTO', 'UPI-UBER-2426@HDFCBANK',
       'POS SWIGGY BANGALORE', 'UPI-GROWWPAY@HDFCBANK', 'OLA ELECTRIC',
       'BMS MOVIE TICKETS', 'POS OLA-PRIME', 'SWIGGY-INSTAMART',
       'UPI-STARBUCKS@AXIS', 'UPI-THIRDWAVE@OKAXIS', 'ANI Technologies',
       'BMTC BUS PASS', 'POS TRUFFLES', 'FLIPKART INDIA',
       'POS SWIGGY-RESTAURANT', 'GROFERS INDIA P L', 'POS UBER BANGALORE',
       'BANGALORE ELEC SUPPLY', 'TWC INDIA', 'UPI-BESCOM-BILL@HDFCBANK',
       'UPI-AMAN-0816@OKAXIS', 'ROPPEN TRANSPORTATION', 'OLA CABS',
       'POS ZOMATO', 'UPI-AMAZONPAY@HDFCBANK', 'POS BLINKIT',
       'IMPS-RENT-LANDLORD-75500265', 'ZOMATO MEDIA P L',
       'UPI-ANKIT-6430@OKAXIS', 'UPI-OLACABS@HDFCBANK',
       'UPI-JIORECHARGE@PAYTM', 'UPI-CCD@HDFCBANK', 'Swiggy*Order',
       'INSTAMART BANGALORE', 'UPI-ZOMATO-LIMITED@PAYTM',
       'AVENUE SUPERMARTS', 'POS HP PETROL

In [92]:
vendor_dict = {

    # Food Delivery
    "Swiggy": ["SWIGGY", "BUNDL"],
    "Zomato": ["ZOMATO"],
    "Truffles": ["TRUFFLES"],
    "Meghana Foods": ["MEGHANA"],
    "Empire Restaurant": ["EMPIRE"],

    # Quick Commerce
    "Zepto": ["ZEPTO"],
    "Blinkit": ["BLINKIT"],

    # E-Commerce
    "Amazon": ["AMAZON", "AMZN"],
    "Flipkart": ["FLIPKART", "FKART"],
    "Myntra": ["MYNTRA"],
    "Nykaa": ["NYKAA"],

    # Transport
    "Uber": ["UBER"],
    "Ola": ["OLA"],
    "Rapido": ["RAPIDO"],
    "BMTC": ["BMTC"],

    # Cafe
    "Starbucks": ["STARBUCKS"],
    "Cafe Coffee Day": ["COFFEE DAY"],
    "Third Wave Coffee": ["THIRD WAVE"],

    # Entertainment
    "BookMyShow": ["BOOKMYSHOW", "BMS"],
    "Netflix": ["NETFLIX"],
    "Spotify": ["SPOTIFY"],
    "Hotstar": ["HOTSTAR"],

    # Investments
    "Zerodha": ["ZERODHA"],
    "Groww": ["GROWW"],

    # Grocery
    "BigBasket": ["BIGBASKET"],
    "DMart": ["DMART"],

    # Fuel
    "Indian Oil": ["INDIAN OIL"],
    "BPCL": ["BPCL"],

    # Utilities
    "Airtel": ["AIRTEL"],
    "Vodafone": ["VODAFONE"],
    "Reliance Jio": ["JIO"],
    "JioFiber": ["JIOFIBER"],
    "BESCOM": ["BESCOM"],
    "BWSSB": ["BWSSB"],

    # Rent
    "Rent": ["LANDLORD", "RENT"],

    # Personal Transfer
    "P2P Transfer": [
        "PRIYA",
        "AMAN",
        "ANKIT",
        "ROHIT",
        "RAHUL",
        "NEHA"
    ],

    # Cash
    "Cash Withdrawal": ["ATM"]
}

In [93]:
def extract_vendor(description):

    description = str(description).upper()

    for vendor, keywords in vendor_dict.items():

        for keyword in keywords:

            if keyword in description:
                return vendor

    return "Uncategorised"

In [98]:
df["vendor_clean"] = df["Description"].apply(extract_vendor)

print("Unique Vendors :", df["vendor_clean"].nunique())

print(df["vendor_clean"].value_counts())

Unique Vendors : 37
vendor_clean
Swiggy               223
Uncategorised        212
Zomato               121
Amazon                86
Uber                  71
Ola                   69
Zepto                 58
Flipkart              47
Starbucks             42
Rapido                41
Blinkit               40
BMTC                  37
Myntra                20
Cash Withdrawal       17
Cafe Coffee Day       17
Nykaa                 16
Zerodha               14
P2P Transfer          13
Empire Restaurant     13
Meghana Foods         12
DMart                 12
Third Wave Coffee     11
BookMyShow            11
BigBasket             11
Indian Oil            11
Netflix               10
Hotstar               10
Groww                  9
Truffles               9
BESCOM                 9
BWSSB                  8
Spotify                8
Rent                   6
Reliance Jio           6
Airtel                 6
BPCL                   2
Vodafone               2
Name: count, dtype: int64


In [95]:
uncategorised = df[df["vendor_clean"]=="Uncategorised"]

print("Total Uncategorised :", len(uncategorised))

uncategorised["Description"].unique()

Total Uncategorised : 212


array(['NEFT-TECHCRUSH LABS-SALARY MAY24', 'UPI-THIRDWAVE@OKAXIS',
       'ANI Technologies', 'GROFERS INDIA P L', 'BANGALORE ELEC SUPPLY',
       'TWC INDIA', 'ROPPEN TRANSPORTATION', 'UPI-CCD@HDFCBANK',
       'INSTAMART BANGALORE', 'AVENUE SUPERMARTS',
       'POS HP PETROL STATION', 'UPI-VIKAS-6060@OKAXIS',
       'POS BANGALORE RESTAURANT', 'VI POSTPAID', 'POS DINEOUT',
       'UPI-IOC9075@HDFCBANK', 'UPI-RESTAURANT-7285@PAYTM',
       'FSN E-COMMERCE', 'UPI-RESTAURANT-7678@PAYTM',
       'BIGTREE ENTERTAINMENT', 'INNOVATIVE RETAIL', 'POS INSTAMART',
       'UPI-RESTAURANT-3715@PAYTM', 'STAR INDIA PVT LTD',
       'UPI-RESTAURANT-0251@PAYTM', 'KIRANAKART TECH',
       'UPI-RESTAURANT-6377@PAYTM', 'UPI-VIKAS-5416@OKAXIS',
       'UPI-IOC0494@HDFCBANK', 'UPI-RESTAURANT-4630@PAYTM',
       'UPI-IOC1246@HDFCBANK', 'UPI-RESTAURANT-7888@PAYTM',
       'UPI-IOC2000@HDFCBANK', 'UPI-VI-RECHARGE@HDFCBANK',
       'UPI-KARAN-5789@OKAXIS', 'UPI-RESTAURANT-0753@PAYTM',
       'UPI-IOC2115@HDFC

In [96]:
#Display Sample Data
df[["Description","vendor_clean"]].head(20)

,Description,vendor_clean
0,AMAZON SELLER SVCS,Amazon
1,BHIM-BMTC,BMTC
2,NEFT-TECHCRUSH LABS-SALARY MAY24,Uncategorised
3,UPI-AMAN-8934@OKAXIS,P2P Transfer
4,BHIM-BLINKIT,Blinkit
5,BHIM ZEPTO,Zepto
6,UPI-UBER-2426@HDFCBANK,Uber
7,BHIM-BLINKIT,Blinkit
8,POS SWIGGY BANGALORE,Swiggy
9,UPI-GROWWPAY@HDFCBANK,Groww


# Feature 3 – Category Tagger

In [99]:
# Create Category Dictionary
category_dict = {

    # Food Delivery
    "Swiggy":"Food Delivery",
    "Zomato":"Food Delivery",

    # Quick Commerce
    "Zepto":"Quick Commerce",
    "Blinkit":"Quick Commerce",

    # E-Commerce
    "Amazon":"E-Commerce",
    "Flipkart":"E-Commerce",
    "Myntra":"E-Commerce",
    "Nykaa":"E-Commerce",

    # Transport
    "Uber":"Transport",
    "Ola":"Transport",
    "Rapido":"Transport",
    "BMTC":"Transport",

    # Cafe
    "Starbucks":"Cafe",
    "Cafe Coffee Day":"Cafe",
    "Third Wave Coffee":"Cafe",

    # Restaurants
    "Empire Restaurant":"Restaurants",
    "Truffles":"Restaurants",
    "Meghana Foods":"Restaurants",

    # Grocery
    "BigBasket":"Groceries",
    "DMart":"Groceries",

    # Investments
    "Zerodha":"Investments",
    "Groww":"Investments",

    # Entertainment
    "BookMyShow":"Entertainment",
    "Netflix":"Subscriptions",
    "Spotify":"Subscriptions",
    "Hotstar":"Subscriptions",

    # Utilities
    "Airtel":"Utilities",
    "Vodafone":"Utilities",
    "Reliance Jio":"Utilities",
    "JioFiber":"Utilities",
    "BESCOM":"Utilities",
    "BWSSB":"Utilities",

    # Fuel
    "Indian Oil":"Fuel",
    "BPCL":"Fuel",

    # Cash
    "Cash Withdrawal":"Cash Withdrawal",

    # Personal
    "P2P Transfer":"Personal Transfer",

    # Rent
    "Rent":"Rent",

    # Unknown
    "Uncategorised":"Others"
}

In [100]:
df["category"] = df["vendor_clean"].map(category_dict)

In [101]:
df["category"] = df["category"].fillna("Others")

In [102]:
print(df["category"].value_counts())

category
Food Delivery        344
Transport            218
Others               212
E-Commerce           169
Quick Commerce        98
Cafe                  70
Restaurants           34
Utilities             31
Subscriptions         28
Groceries             23
Investments           23
Cash Withdrawal       17
Fuel                  13
Personal Transfer     13
Entertainment         11
Rent                   6
Name: count, dtype: int64


In [103]:
df[["Description","vendor_clean","category"]].head(20)

,Description,vendor_clean,category
0,AMAZON SELLER SVCS,Amazon,E-Commerce
1,BHIM-BMTC,BMTC,Transport
2,NEFT-TECHCRUSH LABS-SALARY MAY24,Uncategorised,Others
3,UPI-AMAN-8934@OKAXIS,P2P Transfer,Personal Transfer
4,BHIM-BLINKIT,Blinkit,Quick Commerce
5,BHIM ZEPTO,Zepto,Quick Commerce
6,UPI-UBER-2426@HDFCBANK,Uber,Transport
7,BHIM-BLINKIT,Blinkit,Quick Commerce
8,POS SWIGGY BANGALORE,Swiggy,Food Delivery
9,UPI-GROWWPAY@HDFCBANK,Groww,Investments


In [104]:
print(df["category"].isna().sum())

0


In [105]:
df_clean = df.copy()

# Feature 4 – Spending Overview

In [108]:
#Separate Credits and Debits
credit_df = df[df["Type"]=="credit"]

debit_df = df[df["Type"]=="debit"]

In [109]:
#total_credit = credit_df["Amount"].sum()

print("Total Credits : ₹{:,.2f}".format(total_credit))
total_credit = credit_df["Amount"].sum()

print("Total Credits : ₹{:,.2f}".format(total_credit))

Total Credits : ₹509,774.00
Total Credits : ₹509,774.00


In [111]:
#Calculate Total Debits
total_debit = debit_df["Amount"].sum()

print("Total Debits : ₹{:,.2f}".format(total_debit))

Total Debits : ₹1,678,901.00


In [112]:
#Net Savings
net_savings = total_credit-total_debit

print("Net Savings : ₹{:,.2f}".format(net_savings))

Net Savings : ₹-1,169,127.00


In [113]:
# Savings Rate
savings_rate=(net_savings/total_credit)*100

print("Savings Rate : {:.2f}%".format(savings_rate))

Savings Rate : -229.34%


In [114]:
#Total Transactions
print("Total Transactions :",len(df))

Total Transactions : 1310


In [115]:
#Total Vendors
print("Unique Vendors :",df["vendor_clean"].nunique())

Unique Vendors : 37


In [116]:
top_categories=debit_df.groupby("category")["Amount"].sum()

top_categories=top_categories.sort_values(ascending=False)

print(top_categories.head())

category
E-Commerce       599581.0
Investments      248160.0
Others           193106.0
Food Delivery    150839.0
Rent             108000.0
Name: Amount, dtype: float64


In [117]:
top_vendors=debit_df.groupby("vendor_clean")["Amount"].sum()

top_vendors=top_vendors.sort_values(ascending=False)

print(top_vendors.head())

vendor_clean
Amazon           328530.0
Zerodha          210000.0
Uncategorised    193106.0
Flipkart         177510.0
Rent             108000.0
Name: Amount, dtype: float64


In [118]:
print("="*65)
print("              SPENDING OVERVIEW")
print("="*65)

print(f"Total Credits      : ₹{total_credit:,.2f}")
print(f"Total Debits       : ₹{total_debit:,.2f}")
print(f"Net Savings        : ₹{net_savings:,.2f}")
print(f"Savings Rate       : {savings_rate:.2f}%")
print(f"Transactions       : {len(df)}")
print(f"Unique Vendors     : {df['vendor_clean'].nunique()}")

print("="*65)

              SPENDING OVERVIEW
Total Credits      : ₹509,774.00
Total Debits       : ₹1,678,901.00
Net Savings        : ₹-1,169,127.00
Savings Rate       : -229.34%
Transactions       : 1310
Unique Vendors     : 37


In [119]:
#Print Top Categories
print("\nTOP 5 CATEGORIES")

print("-"*50)

print(top_categories.head())


TOP 5 CATEGORIES
--------------------------------------------------
category
E-Commerce       599581.0
Investments      248160.0
Others           193106.0
Food Delivery    150839.0
Rent             108000.0
Name: Amount, dtype: float64


In [120]:
#Top Vendors
print("\nTOP 5 VENDORS")

print("-"*50)

print(top_vendors.head())


TOP 5 VENDORS
--------------------------------------------------
vendor_clean
Amazon           328530.0
Zerodha          210000.0
Uncategorised    193106.0
Flipkart         177510.0
Rent             108000.0
Name: Amount, dtype: float64


# Feature 5 – Monthly Trend Analysis

In [121]:
# Create Month Column
df["Month"] = df["Date"].dt.month_name()

df[["Date","Month"]].head()

,Date,Month
0,2024-01-01,January
1,2024-01-01,January
2,2024-01-01,January
3,2024-01-01,January
4,2024-01-01,January


In [122]:
#Monthly Spending Pivot Table
monthly_trend = df[df["Type"]=="debit"].pivot_table(
    values="Amount",
    index="category",
    columns="Month",
    aggfunc="sum",
    fill_value=0
)

monthly_trend

Month,April,February,January,June,March,May
category,,,,,,
Cafe,4617.0,3223.0,2194.0,4053.0,3297.0,5058.0
Cash Withdrawal,5500.0,5000.0,2000.0,17000.0,8000.0,8000.0
E-Commerce,69219.0,92738.0,98623.0,137248.0,105977.0,95776.0
Entertainment,1178.0,0.0,1263.0,1914.0,2418.0,0.0
Food Delivery,27756.0,23740.0,22633.0,26499.0,24803.0,25408.0
Fuel,7828.0,1452.0,18791.0,1793.0,19308.0,7188.0
Groceries,4916.0,7517.0,11763.0,1449.0,1865.0,5888.0
Investments,54126.0,15000.0,38432.0,23330.0,68644.0,48628.0
Others,33265.0,27168.0,38629.0,41316.0,31084.0,21644.0


In [123]:
#Arrange Months in Order
month_order = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June"
]

monthly_trend = monthly_trend[month_order]

monthly_trend

Month,January,February,March,April,May,June
category,,,,,,
Cafe,2194.0,3223.0,3297.0,4617.0,5058.0,4053.0
Cash Withdrawal,2000.0,5000.0,8000.0,5500.0,8000.0,17000.0
E-Commerce,98623.0,92738.0,105977.0,69219.0,95776.0,137248.0
Entertainment,1263.0,0.0,2418.0,1178.0,0.0,1914.0
Food Delivery,22633.0,23740.0,24803.0,27756.0,25408.0,26499.0
Fuel,18791.0,1452.0,19308.0,7828.0,7188.0,1793.0
Groceries,11763.0,7517.0,1865.0,4916.0,5888.0,1449.0
Investments,38432.0,15000.0,68644.0,54126.0,48628.0,23330.0
Others,38629.0,27168.0,31084.0,33265.0,21644.0,41316.0


In [124]:
#Print Monthly Trend
print("="*75)
print("MONTHLY SPENDING TREND")
print("="*75)

print(monthly_trend)

MONTHLY SPENDING TREND
Month              January  February     March    April      May      June
category                                                                  
Cafe                2194.0    3223.0    3297.0   4617.0   5058.0    4053.0
Cash Withdrawal     2000.0    5000.0    8000.0   5500.0   8000.0   17000.0
E-Commerce         98623.0   92738.0  105977.0  69219.0  95776.0  137248.0
Entertainment       1263.0       0.0    2418.0   1178.0      0.0    1914.0
Food Delivery      22633.0   23740.0   24803.0  27756.0  25408.0   26499.0
Fuel               18791.0    1452.0   19308.0   7828.0   7188.0    1793.0
Groceries          11763.0    7517.0    1865.0   4916.0   5888.0    1449.0
Investments        38432.0   15000.0   68644.0  54126.0  48628.0   23330.0
Others             38629.0   27168.0   31084.0  33265.0  21644.0   41316.0
Personal Transfer   7523.0    4285.0     266.0    663.0   3412.0    1541.0
Quick Commerce      7051.0   11181.0   12229.0   7509.0   6929.0    6362.0
Re

In [125]:
# Calculate Growth
monthly_trend["Growth"] = (
    (monthly_trend["June"] - monthly_trend["January"])
    / monthly_trend["January"]
) * 100

monthly_trend

/tmp/ipykernel_2912/2254031676.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  monthly_trend["Growth"] = (


Month,January,February,March,April,May,June,Growth
category,,,,,,,
Cafe,2194.0,3223.0,3297.0,4617.0,5058.0,4053.0,84.731085
Cash Withdrawal,2000.0,5000.0,8000.0,5500.0,8000.0,17000.0,750.000000
E-Commerce,98623.0,92738.0,105977.0,69219.0,95776.0,137248.0,39.164292
Entertainment,1263.0,0.0,2418.0,1178.0,0.0,1914.0,51.543943
Food Delivery,22633.0,23740.0,24803.0,27756.0,25408.0,26499.0,17.081253
Fuel,18791.0,1452.0,19308.0,7828.0,7188.0,1793.0,-90.458198
Groceries,11763.0,7517.0,1865.0,4916.0,5888.0,1449.0,-87.681714
Investments,38432.0,15000.0,68644.0,54126.0,48628.0,23330.0,-39.295379
Others,38629.0,27168.0,31084.0,33265.0,21644.0,41316.0,6.955914


In [126]:
#Biggest Growth Category
highest_growth = monthly_trend["Growth"].idxmax()

print("Highest Growth Category :", highest_growth)

print("Growth Percentage : {:.2f}%".format(
    monthly_trend["Growth"].max()
))

Highest Growth Category : Cash Withdrawal
Growth Percentage : 750.00%


In [128]:
#Biggest Decline Category
lowest_growth = monthly_trend["Growth"].idxmin()

print("Highest Decline Category :", lowest_growth)

print("Growth Percentage : {:.2f}%".format(
    monthly_trend["Growth"].min()
))

Highest Decline Category : Fuel
Growth Percentage : -90.46%


In [129]:
#Display Final Table
monthly_trend

Month,January,February,March,April,May,June,Growth
category,,,,,,,
Cafe,2194.0,3223.0,3297.0,4617.0,5058.0,4053.0,84.731085
Cash Withdrawal,2000.0,5000.0,8000.0,5500.0,8000.0,17000.0,750.000000
E-Commerce,98623.0,92738.0,105977.0,69219.0,95776.0,137248.0,39.164292
Entertainment,1263.0,0.0,2418.0,1178.0,0.0,1914.0,51.543943
Food Delivery,22633.0,23740.0,24803.0,27756.0,25408.0,26499.0,17.081253
Fuel,18791.0,1452.0,19308.0,7828.0,7188.0,1793.0,-90.458198
Groceries,11763.0,7517.0,1865.0,4916.0,5888.0,1449.0,-87.681714
Investments,38432.0,15000.0,68644.0,54126.0,48628.0,23330.0,-39.295379
Others,38629.0,27168.0,31084.0,33265.0,21644.0,41316.0,6.955914


In [130]:
df_clean = df.copy()

# Feature 6 – Time-of-Day Analysis

In [131]:
#Create Hour Column
df["Hour"] = df["Time"].str[:2].astype(int)

df[["Time","Hour"]].head()

,Time,Hour
0,03:11,3
1,05:44,5
2,09:35,9
3,14:07,14
4,14:23,14


In [132]:
#Check Spending by Hour
hourly_spending = (
    df[df["Type"]=="debit"]
    .groupby("Hour")["Amount"]
    .sum()
)

print(hourly_spending)

Hour
0      37969.0
1      38882.0
2      20283.0
3      34868.0
4      43901.0
5      33671.0
6      55506.0
7      19938.0
8      41399.0
9      86701.0
10    140526.0
11    123389.0
12     77096.0
13    113819.0
14    121010.0
15     39647.0
16    100597.0
17     56604.0
18    101343.0
19     67848.0
20    138089.0
21     81545.0
22     55867.0
23     48403.0
Name: Amount, dtype: float64


In [133]:
#Find Peak Spending Hour
peak_hour = hourly_spending.idxmax()

peak_amount = hourly_spending.max()

print("Peak Spending Hour :", peak_hour)

print("Amount Spent : ₹{:,.2f}".format(peak_amount))

Peak Spending Hour : 10
Amount Spent : ₹140,526.00


In [134]:
#Category vs Hour Matrix
hour_matrix = df[df["Type"]=="debit"].pivot_table(
    values="Amount",
    index="category",
    columns="Hour",
    aggfunc="sum",
    fill_value=0
)

hour_matrix

Hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
category,,,,,,,,,,,,,,,,,,,,,
Cafe,499.0,348.0,0.0,316.0,0.0,0.0,179.0,480.0,1794.0,1029.0,...,917.0,464.0,2370.0,2377.0,2236.0,372.0,1214.0,0.0,0.0,764.0
Cash Withdrawal,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4000.0,10500.0,...,1000.0,0.0,5000.0,0.0,5000.0,0.0,1000.0,1000.0,4000.0,0.0
E-Commerce,18650.0,9016.0,10246.0,14865.0,12018.0,17258.0,9249.0,10916.0,6537.0,40724.0,...,72584.0,6427.0,52046.0,35378.0,35272.0,12668.0,51190.0,35095.0,16182.0,15403.0
Entertainment,562.0,620.0,0.0,0.0,0.0,0.0,0.0,311.0,1463.0,0.0,...,0.0,0.0,835.0,0.0,1270.0,417.0,952.0,0.0,343.0,0.0
Food Delivery,832.0,3017.0,928.0,1702.0,2618.0,3488.0,238.0,1498.0,3737.0,5299.0,...,4236.0,7891.0,5008.0,3951.0,13587.0,17300.0,18372.0,12123.0,9283.0,4662.0
Fuel,2675.0,0.0,0.0,676.0,2010.0,2105.0,0.0,0.0,0.0,0.0,...,7828.0,10505.0,7707.0,0.0,0.0,0.0,0.0,11845.0,0.0,0.0
Groceries,0.0,1536.0,635.0,7080.0,1169.0,0.0,3645.0,0.0,0.0,4684.0,...,655.0,0.0,0.0,0.0,910.0,0.0,1257.0,2591.0,0.0,2536.0
Investments,4883.0,15000.0,0.0,0.0,18834.0,4496.0,37717.0,0.0,0.0,4737.0,...,0.0,0.0,0.0,0.0,15000.0,4476.0,15000.0,0.0,15000.0,18628.0
Others,6650.0,3034.0,5823.0,6841.0,1796.0,2322.0,3266.0,421.0,13972.0,7586.0,...,4737.0,6796.0,9191.0,8747.0,14885.0,12515.0,20692.0,7916.0,4102.0,4367.0


In [135]:
# Food Delivery Orders by Hour
food = df[df["category"]=="Food Delivery"]

food_hour = food.groupby("Hour").size()

print(food_hour)

Hour
0      2
1      8
2      2
3      4
4      7
5      7
6      1
7      3
8      9
9     13
10     7
11    21
12    27
13    15
14     9
15    17
16    10
17    11
18    30
19    38
20    42
21    27
22    24
23    10
dtype: int64


In [136]:
#Late-Night Orders
late_night = food[
    (food["Hour"]>=21) | (food["Hour"]<=2)
]

print("Late Night Orders :", len(late_night))

Late Night Orders : 73


In [137]:
#Total Food Orders
print("Total Food Orders :", len(food))

Total Food Orders : 344


In [138]:
#Percentage of Late-Night Orders
late_percentage = (
    len(late_night) / len(food)
) * 100

print("Late Night Percentage : {:.2f}%".format(late_percentage))

Late Night Percentage : 21.22%


In [139]:
print("="*70)
print("TIME OF DAY ANALYSIS")
print("="*70)

print(f"Peak Spending Hour : {peak_hour}:00")
print(f"Highest Spending   : ₹{peak_amount:,.2f}")

print()

print(f"Food Delivery Orders : {len(food)}")
print(f"Late Night Orders    : {len(late_night)}")
print(f"Late Night %         : {late_percentage:.2f}%")

TIME OF DAY ANALYSIS
Peak Spending Hour : 10:00
Highest Spending   : ₹140,526.00

Food Delivery Orders : 344
Late Night Orders    : 73
Late Night %         : 21.22%


# Feature 7 – Anomaly Detection

In [141]:
#Calculate Category Mean
category_mean = df.groupby("category")["Amount"].transform("mean")

In [143]:
#Calculate Category Standard Deviation
category_std = df.groupby("category")["Amount"].transform("std")

In [144]:
# Calculate Z-Score
df["z_score"] = (
    (df["Amount"] - category_mean)
    / category_std
)

In [145]:
# Find Anomalies
anomalies = df[df["z_score"] > 2]

In [146]:
# Number of Anomalies
print("Total Anomalies :", len(anomalies))

Total Anomalies : 31


In [147]:
# Top 10 Anomalies
top_anomalies = anomalies.sort_values(
    by="z_score",
    ascending=False
)

top_anomalies.head(10)

,Date,Time,Description,Type,Amount,Balance,Mode,Ref,vendor_clean,category,Month,Hour,z_score
1113,2024-06-01,09:31,NEFT-TECHCRUSH LABS-SALARY MAY24,credit,85492.0,720079.0,NEFT,TXN399607,Uncategorised,Others,June,9,5.861025
880,2024-05-01,09:37,NEFT-TECHCRUSH LABS-SALARY MAY24,credit,85393.0,671166.0,NEFT,TXN820221,Uncategorised,Others,May,9,5.853964
658,2024-04-01,09:23,NEFT-TECHCRUSH LABS-SALARY MAY24,credit,84736.0,623971.0,NEFT,TXN395442,Uncategorised,Others,April,9,5.807106
2,2024-01-01,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,credit,84728.0,484728.0,NEFT,TXN246316,Uncategorised,Others,January,9,5.806535
224,2024-02-01,09:42,NEFT-TECHCRUSH LABS-SALARY MAY24,credit,84724.0,530245.0,NEFT,TXN542666,Uncategorised,Others,February,9,5.806250
438,2024-03-01,09:35,NEFT-TECHCRUSH LABS-SALARY MAY24,credit,84701.0,576808.0,NEFT,TXN214576,Uncategorised,Others,March,9,5.804609
1298,2024-06-26,09:07,AMAZONIN MARKETPLACE,debit,22008.0,-699028.0,UPI,TXN313051,Amazon,E-Commerce,June,9,4.054513
269,2024-02-07,17:28,UPI-AMAZONPAY@HDFCBANK,debit,21986.0,-360151.0,UPI,TXN988380,Amazon,E-Commerce,February,17,4.049681
653,2024-03-31,20:01,POS MEGHANA FOODS,debit,7931.0,-445522.0,UPI,TXN149269,Meghana Foods,Restaurants,March,20,3.947735
464,2024-03-04,16:12,POS TRUFFLES,debit,7441.0,-472880.0,UPI,TXN830484,Truffles,Restaurants,March,16,3.644259


In [148]:
top_anomalies[
    [
        "Date",
        "Description",
        "vendor_clean",
        "category",
        "Amount",
        "z_score"
    ]
].head(10)

,Date,Description,vendor_clean,category,Amount,z_score
1113,2024-06-01,NEFT-TECHCRUSH LABS-SALARY MAY24,Uncategorised,Others,85492.0,5.861025
880,2024-05-01,NEFT-TECHCRUSH LABS-SALARY MAY24,Uncategorised,Others,85393.0,5.853964
658,2024-04-01,NEFT-TECHCRUSH LABS-SALARY MAY24,Uncategorised,Others,84736.0,5.807106
2,2024-01-01,NEFT-TECHCRUSH LABS-SALARY MAY24,Uncategorised,Others,84728.0,5.806535
224,2024-02-01,NEFT-TECHCRUSH LABS-SALARY MAY24,Uncategorised,Others,84724.0,5.806250
438,2024-03-01,NEFT-TECHCRUSH LABS-SALARY MAY24,Uncategorised,Others,84701.0,5.804609
1298,2024-06-26,AMAZONIN MARKETPLACE,Amazon,E-Commerce,22008.0,4.054513
269,2024-02-07,UPI-AMAZONPAY@HDFCBANK,Amazon,E-Commerce,21986.0,4.049681
653,2024-03-31,POS MEGHANA FOODS,Meghana Foods,Restaurants,7931.0,3.947735
464,2024-03-04,POS TRUFFLES,Truffles,Restaurants,7441.0,3.644259


In [149]:
print("="*70)
print("ANOMALY DETECTION")
print("="*70)

print("Total Anomalies :", len(anomalies))

print()

print(
    top_anomalies[
        [
            "Date",
            "vendor_clean",
            "category",
            "Amount",
            "z_score"
        ]
    ].head(10)
)

ANOMALY DETECTION
Total Anomalies : 31

           Date   vendor_clean     category   Amount   z_score
1113 2024-06-01  Uncategorised       Others  85492.0  5.861025
880  2024-05-01  Uncategorised       Others  85393.0  5.853964
658  2024-04-01  Uncategorised       Others  84736.0  5.807106
2    2024-01-01  Uncategorised       Others  84728.0  5.806535
224  2024-02-01  Uncategorised       Others  84724.0  5.806250
438  2024-03-01  Uncategorised       Others  84701.0  5.804609
1298 2024-06-26         Amazon   E-Commerce  22008.0  4.054513
269  2024-02-07         Amazon   E-Commerce  21986.0  4.049681
653  2024-03-31  Meghana Foods  Restaurants   7931.0  3.947735
464  2024-03-04       Truffles  Restaurants   7441.0  3.644259


In [150]:
highest = top_anomalies.iloc[0]

print()

print("Highest Anomaly")

print("-"*40)

print("Vendor :", highest["vendor_clean"])

print("Category :", highest["category"])

print("Amount : ₹{:,.2f}".format(highest["Amount"]))

print("Z Score :", round(highest["z_score"],2))


Highest Anomaly
----------------------------------------
Vendor : Uncategorised
Category : Others
Amount : ₹85,492.00
Z Score : 5.86


# Feature 8 : Spending Archetype Detection

In [151]:
# Calculate Total Debit
total_debit = debit_df["Amount"].sum()

In [152]:
# Food Percentage
food_total = debit_df[
    debit_df["category"].isin(
        ["Food Delivery","Restaurants","Cafe"]
    )
]["Amount"].sum()

food_percentage = (food_total / total_debit) * 100

In [153]:
# Quick Commerce Percentage
quick_total = debit_df[
    debit_df["category"]=="Quick Commerce"
]["Amount"].sum()

quick_percentage = (quick_total / total_debit) * 100

In [154]:
# E-Commerce Percentage
ecommerce_total = debit_df[
    debit_df["category"]=="E-Commerce"
]["Amount"].sum()

ecommerce_percentage = (ecommerce_total / total_debit) * 100

In [156]:
# Investment Percentage
investment_total = debit_df[
    debit_df["category"]=="Investments"
]["Amount"].sum()

investment_percentage = (investment_total / total_debit) * 100

In [157]:
# Transport Percentage
transport_total = debit_df[
    debit_df["category"]=="Transport"
]["Amount"].sum()

transport_percentage = (transport_total / total_debit) * 100

In [158]:
# Subscription Vendors
subscription_vendors = df[
    df["category"]=="Subscriptions"
]["vendor_clean"].nunique()

In [159]:
# Detect Archetypes
archetypes = []

if food_percentage > 25:
    archetypes.append("THE FOODIE")

if quick_percentage > 15:
    archetypes.append("THE QUICK COMMERCE JUNKIE")

if ecommerce_percentage > 15:
    archetypes.append("THE SHOPAHOLIC")

if investment_percentage > 15:
    archetypes.append("THE INVESTOR")

if late_percentage > 50:
    archetypes.append("THE LATE-NIGHT SNACKER")

if transport_percentage > 10:
    archetypes.append("THE CAB COMMUTER")

if subscription_vendors >= 3:
    archetypes.append("THE SUBSCRIPTION LOVER")

if savings_rate < 10:
    archetypes.append("THE YOLO SPENDER")

if savings_rate > 40:
    archetypes.append("THE DISCIPLINED SAVER")

In [160]:
# Display Archetypes
print("="*70)
print("SPENDING ARCHETYPES")
print("="*70)

for a in archetypes:
    print("->", a)

SPENDING ARCHETYPES
-> THE SHOPAHOLIC
-> THE SUBSCRIPTION LOVER
-> THE YOLO SPENDER


In [162]:
print("\nFood Percentage :", round(food_percentage,2),"%")
print("Quick Commerce :", round(quick_percentage,2),"%")
print("E-Commerce :", round(ecommerce_percentage,2),"%")
print("Investment :", round(investment_percentage,2),"%")
print("Transport :", round(transport_percentage,2),"%")
print("Late Night :", round(late_percentage,2),"%")
print("Subscription Vendors :", subscription_vendors)
print("Savings Rate :", round(savings_rate,2),"%")


Food Percentage : 13.47 %
Quick Commerce : 3.05 %
E-Commerce : 35.71 %
Investment : 14.78 %
Transport : 2.99 %
Late Night : 21.22 %
Subscription Vendors : 3
Savings Rate : -229.34 %


# Final SpendDNA Report

In [165]:
print("="*80)
print("                     SPENDDNA REPORT")
print("="*80)
print("                Mohammad Dhilawala")
print("         6 Months Financial Transaction Analysis")
print("="*80)

print("\nEXECUTIVE SUMMARY")
print("-"*80)
print(f"Total Credits      : ₹{total_credit:,.2f}")
print(f"Total Debits       : ₹{total_debit:,.2f}")
print(f"Net Savings        : ₹{net_savings:,.2f}")
print(f"Savings Rate       : {savings_rate:.2f}%")
print(f"Transactions       : {len(df)}")
print(f"Unique Vendors     : {df['vendor_clean'].nunique()}")

print("\nTOP 5 CATEGORIES")
print("-"*80)
print(top_categories.head())

print("\nTOP 5 VENDORS")
print("-"*80)
print(top_vendors.head())

print("\nTIME-OF-DAY PATTERNS")
print("-"*80)
print(f"Food Delivery orders 9 PM - 2 AM : {late_percentage:.1f}%")
print(f"Cafe spending peaks around       : {peak_hour}:00")

print("\nMONTHLY TREND (E-Commerce)")
print("-"*80)

ecommerce = monthly_trend.loc["E-Commerce"]

months = ["January","February","March","April","May","June"]
short = ["Jan","Feb","Mar","Apr","May","Jun"]

for s, m in zip(short, months):
    value = ecommerce[m]
    bar = "#" * int(value / 4000)   # Adjust bar length if needed
    print(f"{s:<5} Rs. {value:>8,.0f}  {bar}")

print("\nTOP ANOMALIES (z-score > 2 within category)")
print("-"*80)

for _, row in top_anomalies.head(5).iterrows():
    date = row["Date"].strftime("%d %b")
    vendor = row["vendor_clean"]
    amount = row["Amount"]
    z = row["z_score"]
    print(f"{date} - {vendor:<15} Rs. {amount:>8,.0f} (z={z:.1f})")

print("\nSPENDING ARCHETYPES")
print("-"*80)

for a in archetypes:
    print("✔", a)

print("="*80)
print("                    END OF REPORT")
print("="*80)

                     SPENDDNA REPORT
                Mohammad Dhilawala
         6 Months Financial Transaction Analysis

EXECUTIVE SUMMARY
--------------------------------------------------------------------------------
Total Credits      : ₹509,774.00
Total Debits       : ₹1,678,901.00
Net Savings        : ₹-1,169,127.00
Savings Rate       : -229.34%
Transactions       : 1310
Unique Vendors     : 37

TOP 5 CATEGORIES
--------------------------------------------------------------------------------
category
E-Commerce       599581.0
Investments      248160.0
Others           193106.0
Food Delivery    150839.0
Rent             108000.0
Name: Amount, dtype: float64

TOP 5 VENDORS
--------------------------------------------------------------------------------
vendor_clean
Amazon           328530.0
Zerodha          210000.0
Uncategorised    193106.0
Flipkart         177510.0
Rent             108000.0
Name: Amount, dtype: float64

TIME-OF-DAY PATTERNS
--------------------------------------

# KEY INSIGHTS

In [167]:
debit_credit_ratio = total_debit / total_credit

total_spend = debit_df["Amount"].sum()

top_cat_name = top_categories.index[0]
top_cat_pct = (top_categories.iloc[0] / total_spend) * 100

invest_pct = (
    debit_df[debit_df["category"] == "Investments"]["Amount"].sum()
    / total_spend
) * 100

print("KEY INSIGHTS")
print("-" * 60)

print(f"1. Rahul's total debits are {debit_credit_ratio:.1f} times higher than his total credits over the six-month period.")
print(f"   With a savings rate of {savings_rate:.1f}%, he is rapidly depleting")
print(f"   his savings. If this spending pattern continues,")
print(f"   his financial reserves may reduce significantly.")

print()

print(f"2. {top_cat_name} is the largest spending category, accounting for")
print(f"   {top_cat_pct:.1f}% of total spending, making it the")
print(f"   biggest contributor to overall expenses.")

print()

print(f"3. Investments still account for {invest_pct:.1f}% of total spending,")
print(f"   indicating consistent investment habits. However,")
print(f"   discretionary expenses continue to dominate the overall expenditure.")

KEY INSIGHTS
------------------------------------------------------------
1. Rahul's total debits are 3.3 times higher than his total credits over the six-month period.
   With a savings rate of -229.3%, he is rapidly depleting
   his savings. If this spending pattern continues,
   his financial reserves may reduce significantly.

2. E-Commerce is the largest spending category, accounting for
   35.7% of total spending, making it the
   biggest contributor to overall expenses.

3. Investments still account for 14.8% of total spending,
   indicating consistent investment habits. However,
   discretionary expenses continue to dominate the overall expenditure.


# Reflection

This project helped me understand how real-world financial transaction data can be cleaned, processed, and analyzed using Python, Pandas, and NumPy.

During the project I learned:

- Data Cleaning
- Handling multiple date formats
- Cleaning currency values
- Vendor extraction
- Transaction categorization
- Monthly trend analysis
- Time-based spending analysis
- Z-score anomaly detection
- Spending archetype detection

This project improved my data analysis, data preprocessing, and problem-solving skills while working with a realistic financial dataset.